# Feature Engineering

This notebook will focus on engineering new features derived from existing features in the Djokovic dataset formed in the cleaning notebook, as well as restructuring certain features to allow for better compatability with many of the models we will be using.

### 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas.api.types as ptypes # for data type querying

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams.update({
    "font.size": 14,     
    "axes.titlesize": 18,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
})

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "last_expr"


### 2. Load Data

In [2]:
df = pd.read_csv("../Data/djokovic_data.csv")
print(f"Shape of dataset {df.shape}")
df.head(10)

Shape of dataset (1250, 15)


,Date,Series,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2
0,2005-07-26,International,Outdoor,Clay,1st Round,3,Djokovic N.,Calatrava A.,Djokovic N.,97,163,417,259,1.36,2.87
1,2005-07-27,International,Outdoor,Clay,2nd Round,3,Ferrero J.C.,Djokovic N.,Ferrero J.C.,28,97,1170,417,1.14,5.00
2,2005-08-15,Masters,Outdoor,Hard,1st Round,3,Djokovic N.,Gonzalez F.,Gonzalez F.,97,18,440,1385,3.50,1.28
3,2005-08-30,Grand Slam,Outdoor,Hard,1st Round,5,Djokovic N.,Monfils G.,Djokovic N.,97,43,431,825,2.50,1.50
4,2005-09-02,Grand Slam,Outdoor,Hard,2nd Round,5,Djokovic N.,Ancic M.,Djokovic N.,97,24,431,1230,3.50,1.28
5,2005-09-04,Grand Slam,Outdoor,Hard,3rd Round,5,Djokovic N.,Verdasco F.,Verdasco F.,97,48,431,770,2.75,1.39
6,2005-10-24,International,Indoor,Carpet,1st Round,3,Djokovic N.,Mello R.,Djokovic N.,88,105,468,398,1.39,2.75
7,2005-10-26,International,Indoor,Carpet,2nd Round,3,Djokovic N.,Rochus O.,Rochus O.,88,24,468,1160,3.00,1.36
8,2005-11-01,Masters,Indoor,Carpet,2nd Round,3,Puerta M.,Djokovic N.,Djokovic N.,9,85,1834,488,2.20,1.61
9,2005-11-03,Masters,Indoor,Carpet,3rd Round,3,Djokovic N.,Robredo T.,Robredo T.,85,21,488,1370,2.37,1.53


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1250 entries, 0 to 1249
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      1250 non-null   str    
 1   Series    1250 non-null   str    
 2   Court     1250 non-null   str    
 3   Surface   1250 non-null   str    
 4   Round     1250 non-null   str    
 5   Best of   1250 non-null   int64  
 6   Player_1  1250 non-null   str    
 7   Player_2  1250 non-null   str    
 8   Winner    1250 non-null   str    
 9   Rank_1    1250 non-null   int64  
 10  Rank_2    1250 non-null   int64  
 11  Pts_1     1250 non-null   int64  
 12  Pts_2     1250 non-null   int64  
 13  Odd_1     1250 non-null   float64
 14  Odd_2     1250 non-null   float64
dtypes: float64(2), int64(5), str(8)
memory usage: 146.6 KB


In [4]:
df.describe().round(2)

,Best of,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2
count,1250.00,1250.00,1250.00,1250.00,1250.00,1250.00,1250.00
mean,3.68,27.01,25.93,5362.00,5699.71,5.27,4.98
std,0.95,46.97,67.86,4625.39,4707.07,6.54,6.66
min,3.00,1.00,1.00,43.00,1.00,1.00,1.00
25%,3.00,2.00,2.00,1132.75,1327.50,1.12,1.10
50%,3.00,6.00,5.00,4067.50,4445.00,2.30,1.61
75%,5.00,35.75,27.75,9193.75,9857.50,7.00,6.00
max,5.00,589.00,1543.00,16950.00,16950.00,34.00,41.00


## 2. Points and Rank wrangling

Currently, there are two points columns and two rank columns, one for each player. These are filled by which player was given the title player $1$ and which player was given the title player $2$. Hence, as prediction columns they are ambiguous, as a model is unable to distinguish which player has which odds and rank purely from this, and in particular will not know which ranks and points belong to Djokovic. We will instead create new columns for Djokovic's points and rank, and columns for the opponents.

In [5]:
# Forming columns for Djokovic points/rank/odds and opponent's

dj_is_1 = df["Player_1"] == "Djokovic N."

df["dj_pts"] = np.where(dj_is_1, df["Pts_1"], df["Pts_2"])
df["dj_rank"] = np.where(dj_is_1, df["Rank_1"], df["Rank_2"])
df["dj_odds"] = np.where(dj_is_1, df["Odd_1"], df["Odd_2"])

df["opp_pts"] = np.where(dj_is_1, df["Pts_2"], df["Pts_1"])
df["opp_rank"] = np.where(dj_is_1, df["Rank_2"], df["Rank_1"])
df["opp_odds"] = np.where(dj_is_1, df["Odd_2"], df["Odd_1"])

Now that we have formed these new columns, the old points/rank/odds columns are unnecessary, so we will drop them. We also drop the Player 1 / Player 2 columns since we always know Djokovic is playing, and the opponent's name is too high-cardinality to effectively encode and their strength is already encoded through their rank and points. Many of our models will require one-hot encoding for categorical data, as we cannot just give them ordered integer labels since this invents an order of magnitude for the data when there isn't one, and many models such as logistic regression depend on the magnitude of column entries. If we one-hot encoded the player columns, we would have a huge number of additional columns with mainly zeros, and some models may overfit this badly.

In [6]:
drop_cols = ["Player_1", "Player_2", "Pts_1", "Pts_2", "Rank_1", "Rank_2", "Odd_1", "Odd_2"]
df_3 = df.copy().drop(columns=drop_cols)

In [7]:
df_3.columns

Index(['Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Winner',
       'dj_pts', 'dj_rank', 'dj_odds', 'opp_pts', 'opp_rank', 'opp_odds'],
      dtype='str')

## 3. Win Wrangling

Currently, we have a column giving the name of the winner as a string. This is useful when we are not considering just the matches of one player. However, since we are only considering Djokovic's matches, we will change this to a binary $0$/$1$ column for Djokovic winning/losing. This encodes exactly the same information but makes it more accessible as target variable for classification tasks.

In [8]:
df_3['dj_win'] = np.where(df_3["Winner"] == "Djokovic N.", 1, 0)

In [9]:
print(df_3['dj_win'].isin([0, 1]).all())

True


In [10]:
df_4 = df_3.copy().drop(columns="Winner")

## 4. Date Wrangling

Using raw date date gives overly complex information of the exact date of the matches that is not informative. Knowing the exact day in the month of the match is unnecessary. Knowing the month can be useful, but has redundancy with the surface column since the surfaces are arranged temporally. Knowing the year / Djokovic's age does encode some important information though, since for obvious reasons a player's age will affect the probability of the player winning the match. We will drop the day and month information, and form an age column. We also form an $age^2$ column since there is often a quadratic relationship between an athlete's age and their performance.

In [11]:
dates = pd.to_datetime(df_4["Date"])
df_4["age"] = (dates - pd.Timestamp("1987-05-22")).dt.days / 365.25
df_4["age_c"] = df_4["age"] - 28
df_4["age_c_sq"] = df_4["age_c"] ** 2
df_4.drop(columns="age", inplace=True)

In [12]:
df_5 = df_4.copy().drop(columns='Date')

In [13]:
df_5.columns

Index(['Series', 'Court', 'Surface', 'Round', 'Best of', 'dj_pts', 'dj_rank',
       'dj_odds', 'opp_pts', 'opp_rank', 'opp_odds', 'dj_win', 'age_c',
       'age_c_sq'],
      dtype='str')

## 5. Outdoor/Indoor, 3/5 Set Encoding

Currently, the 'Court' and 'Best of' columns gives Outdoor/Indoor labels for the setting of the court for the match. We will change these to a binary $0$/$1$ label, which encodes the exact same information and makes it easier for models to understand.

In [14]:
is_outdoor = df_5["Court"] == "Outdoor"
is_five = df_5["Best of"] == 5

df_5["is_outdoor"] = is_outdoor.astype(int)
df_5["is_five_sets"] = is_five.astype(int)

In [15]:
df_5['is_outdoor'].unique()

array([1, 0])

In [16]:
df_5["is_five_sets"].unique()

array([0, 1])

Now we drop the existing "Court" and "Best of" columns since they has been made redundant.

In [17]:
df_6 = df_5.copy().drop(columns=["Court", "Best of"])

In [18]:
df_6.columns

Index(['Series', 'Surface', 'Round', 'dj_pts', 'dj_rank', 'dj_odds', 'opp_pts',
       'opp_rank', 'opp_odds', 'dj_win', 'age_c', 'age_c_sq', 'is_outdoor',
       'is_five_sets'],
      dtype='str')

## 6. Moving Average

We now create a simple $5$-match moving average column. For the first $5$ rows, we use whatever number of matches is available to give an average.

In [19]:
five_sma_col = pd.Series(np.nan, index=df_6.index)

for i in range(five_sma_col.size):
    if i <= 5:
        five_sma_col.iloc[i] = df_6.iloc[:i]['dj_win'].mean()
    else:
        five_sma_col.iloc[i] = df_6.iloc[i-5:i]['dj_win'].mean()

df_6["5SMA"] = five_sma_col

In [20]:
df_6.head(5)["5SMA"]

0         NaN
1    1.000000
2    0.500000
3    0.333333
4    0.500000
Name: 5SMA, dtype: float64

Since the very first row/match does not have any previous matches to calculate this moving average from, it will have an NaN entry. Since this can break some models, and we already have thousands of datapoints to learn from, we will just drop this row.

In [21]:
df_7 = df_6.iloc[1:]

In [22]:
assert df_7["5SMA"].notna().all()

## 7. Carpet and Series Handling

We notice that there are some entries in the surface and series columns that are inconsistent and need handling. Firstly, there are a small number of matches with "Carpet" as the surface, but none after a certain date. When doing our chronological train/test split, this will mean we will train on data that will never appear in the test set. We will drop these rows.

Next, the names of series changed in $2009$. International Series Gold became ATP500 and International Series became ATP250. If we leave out data as is, then we will have different categories actually describing the same series, but the model will treat them as separate. We need to merge these into one category.

In [23]:
df_8 = df_7[df_7["Surface"] != "Carpet"]

df_8["Surface"].unique()

<StringArray>
['Clay', 'Hard', 'Grass']
Length: 3, dtype: str

In [24]:
print(f"Number of rows: {df_8.shape[0]}")

Number of rows: 1238


In [25]:
df_8["Series"].unique()

<StringArray>
[     'International',            'Masters',         'Grand Slam',
 'International Gold',        'Masters Cup',             'ATP250',
             'ATP500',       'Masters 1000']
Length: 8, dtype: str

In [26]:
mapping = {
    "International" : "ATP250", "International Gold" : "ATP500",
    "Masters" : "Masters 1000"}
df_8["Series"] = df_8["Series"].replace(mapping)

In [27]:
df_8["Series"].value_counts()

Series
Masters 1000    500
Grand Slam      423
ATP500          128
ATP250          122
Masters Cup      65
Name: count, dtype: int64

# 8. Different Datasets

Finally, the main goal of this project is to analyse how models perform when trained on our features versus when trained only on odds features. Since odds are bookmaker-produced probabilities of a player winning, they should be very good indicators of the the true win/loss probability. Hence, we expect that most of the explanatory power in our dataset will come from the odds features, and that the other features will just attempt to close the gap between bookmaker odds and true odds as much as they can.

In order to do this, we form three datasets: one with only odds features, one with all features except odds, and one with all features including odds. This will allow us to analyse the discrepancy in model performance that occurs due to the different training features.

For each of these three datasets, we will create one dataset with one-hot encoded categorical features, and one without, so that we can still do effective and clear EDA using the usual data, and so that the models can correctly interpret the information using the one-hot data. Careful consideration of perfect collinearity with the intercept term needs to be done, and we solve this by dropping one of the categorical columns/variables from the one-hot-encoded data. This ensures that this collinearity doesn't occur and we never have a singular data matrix.

In [28]:
encoded = pd.concat([pd.get_dummies(df_8["Series"]).drop(columns="ATP250").astype(int),
                    pd.get_dummies(df_8["Surface"]).drop(columns="Hard").astype(int),
                    pd.get_dummies(df_8["Round"]).drop(columns="1st Round").astype(int)], axis=1)

In [29]:
df_8_encoded = pd.concat([df_8, encoded], axis=1)

In [30]:
df_8_encoded = df_8_encoded.copy().drop(columns=["Series", "Surface", "Round"])

In [31]:
# Full dataset
df_8.to_csv("../Data/full_data.csv", index=False)

# Encoded
df_8_encoded.to_csv("../Data/full_data_enc.csv", index=False)

In [32]:
# Extracting odds only
df_odds = df_8[["dj_odds", "opp_odds", "dj_win"]]
df_odds_encoded = df_8_encoded[["dj_odds", "opp_odds", "dj_win"]]

df_odds.to_csv("../Data/odds_data.csv", index=False)
df_odds_encoded.to_csv("../Data/odds_data_enc.csv", index=False)

In [33]:
# Extracting no odds data
df_nodds = df_8.copy().drop(columns=["dj_odds", "opp_odds"])
df_nodds_enc = df_8_encoded.copy().drop(columns=["dj_odds", "opp_odds"])

df_nodds.to_csv("../Data/nodds_data.csv", index=False)
df_nodds_enc.to_csv("../Data/nodds_data_enc.csv", index=False)